# Which amount band is the model worst at?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [13]:
import os

import polars as pl
from google.cloud import bigquery, storage
from sklearn.metrics import average_precision_score, roc_auc_score

from fraud_detection.core.feature_contract.admission import load_admission_rules
from fraud_detection.core.promotion import parse_promotion_marker
from fraud_detection.core.schema import MODEL_INPUT_TABLE, SPLIT_TABLE
from fraud_detection.feature_engineering.derivations import apply_derivations
from fraud_detection.training.data import load_raw_split, prepare_features, to_lightgbm

PROJECT = os.environ["GCP_PROJECT_ID"]
bq = bigquery.Client(project=PROJECT)
gcs = storage.Client(project=PROJECT).bucket(f"{PROJECT}-models")

In [10]:
# The promoted model, read the way the scoring path reads it — the marker, never the
# newest artifact. Analysing a model nobody promoted would describe a decision nobody made.
import pickle

promoted = parse_promotion_marker(gcs.blob("promoted/production.json").download_as_text())
print(f"{promoted.run}  contract {promoted.contract_fingerprint}  code {promoted.code_version[:12]}")

bundle = pickle.loads(gcs.blob(f"lightgbm/{promoted.run}/model.pkl").download_as_bytes())
booster = bundle["booster"]

raw = load_raw_split(bq, PROJECT, "test", model_input_table=MODEL_INPUT_TABLE, split_table=SPLIT_TABLE)
derived = apply_derivations(raw, load_admission_rules().derivations)
features = prepare_features(derived)
scores = booster.predict(to_lightgbm(features.select(booster.feature_name())),
                         num_iteration=booster.best_iteration)

frame = raw.select(["TransactionID", "isFraud", "TransactionAmt", "TransactionDT",
                    "ProductCD", "card1", "addr1", "D1", "D9"]).with_columns(
    score=pl.Series(scores)
)
print(frame.shape)

ef077ec2  contract bdb97707da05adff  code 89145b690b7b
(59054, 10)


## 1. Where the errors are

One helper, applied to every cut. **PR-AUC per segment, not accuracy** — with a 3.5% base
rate, accuracy is a measure of how many rows are negative.

A segment's PR-AUC is not comparable to another segment's if their base rates differ, so
the lift over each segment's *own* base rate is the column to read.

In [4]:
MIN_ROWS, MIN_POSITIVES = 500, 20


def by_segment(df: pl.DataFrame, column: str) -> pl.DataFrame:
    """PR-AUC within each level of `column`, with its own base rate beside it.

    Segments too small to estimate are reported as null rather than dropped: a segment
    nobody can measure is a finding about coverage, and silently omitting it would make
    the table look more complete than the data is.
    """
    rows = []
    for (level,), group in df.group_by([column], maintain_order=True):
        y, s = group["isFraud"].to_numpy(), group["score"].to_numpy()
        base = float(y.mean())
        measurable = len(group) >= MIN_ROWS and y.sum() >= MIN_POSITIVES
        pr = float(average_precision_score(y, s)) if measurable else None
        rows.append({
            column: level, "rows": len(group), "positives": int(y.sum()),
            "base_rate": round(base, 4),
            "pr_auc": None if pr is None else round(pr, 4),
            "lift_over_base": None if pr is None or base == 0 else round(pr / base, 2),
        })
    return pl.DataFrame(rows).sort("rows", descending=True)


overall = average_precision_score(frame["isFraud"].to_numpy(), frame["score"].to_numpy())
print(f"overall test PR-AUC {overall:.4f}, ROC-AUC "
      f"{roc_auc_score(frame['isFraud'].to_numpy(), frame['score'].to_numpy()):.4f}")

overall test PR-AUC 0.5308, ROC-AUC 0.8963
